In [2]:
import sys 
print(sys.executable)
import torch

import warnings
import tokenizers
import accelerate
import transformers
from transformers import AutoModelForQuestionAnswering, AutoTokenizer,AutoConfig, AutoModelForCausalLM

import peft
from peft import PeftModel
import torch

import torch.nn.functional as F
# import torchvision.transforms as transforms
# import torchvision.models as models

import captum
from captum.attr import IntegratedGradients, Occlusion, LayerGradCam, LayerAttribution
from captum.attr import visualization as viz

print("torch version: ", torch.__version__)
print("accelerate version: ", accelerate.__version__)
print("peft version: ", peft.__version__)
print("transformers version: ", transformers.__version__)

/home/vasco/nguyehmk/miniconda3/envs/llm/bin/python
False
'CUDASetup' object has no attribute 'cuda_available'


/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


torch version:  2.0.1+cu117
accelerate version:  0.28.0
peft version:  0.10.0
transformers version:  4.38.2


In [3]:
import psutil
import platform

print("===== SYSTEM INFO =====")
print("OS:", platform.system(), platform.release())
print("Processor:", platform.processor())
print("CPU cores:", psutil.cpu_count())

ram = psutil.virtual_memory()
print("RAM:", round(ram.total / (1024**3), 2), "GB")

print("\n===== GPU INFO =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

===== SYSTEM INFO =====
OS: Linux 6.1.0-45-amd64
Processor: 
CPU cores: 32
RAM: 251.67 GB

===== GPU INFO =====
CUDA available: True
GPU: NVIDIA GeForce GTX 1080 Ti
VRAM: 10.91 GB


In [4]:
# Load model directly
torch.cuda.empty_cache()

# Check device availability
device = torch.device("cuda")
print("Model device:", device)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B")

# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B", device_map="auto", torch_dtype="auto").to(device)

print("Load model completed")

Model device: cuda


/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Load model completed


In [ ]:
# Code block for question - answering in original model

messages = [
    {"role": "user", "content": "Write a python function for calculating the factorial of an integer."},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

outputs = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [5]:
base_model_name = "Qwen/Qwen2.5-Coder-3B"  

torch.cuda.empty_cache()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load base model (with quantization if needed)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto"
)

# Attach your fine-tuned adapters
model = PeftModel.from_pretrained(base_model, "model_ft")

print("load done")

device = torch.device('cuda:0')  # Specify GPU device
print(f"Total memory: {torch.cuda.get_device_properties(device).total_memory / 1024**3:.2f} GB")
print(f"Allocated memory: {torch.cuda.memory_allocated(device) / 1024**3:.2f} GB")
print(f"Reserved memory: {torch.cuda.memory_reserved(device) / 1024**3:.2f} GB")

# Optional: improve inference speed
model.to(device)
model.eval()

# Prompt
input_text = "Write unit tests for a function that adds two numbers"

inputs = tokenizer(
    input_text,
    return_tensors="pt"
).to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/accelerate/utils/modeling.py:1341: UserWarning: Current model requires 1107304704 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.22it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB (GPU 0; 10.91 GiB total capacity; 2.93 GiB already allocated; 4.06 MiB free; 2.93 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [6]:


base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B",
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base,
    "./model_ft"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B"
)

/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/accelerate/utils/modeling.py:1341: UserWarning: Current model requires 1207968768 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.45it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB (GPU 0; 10.91 GiB total capacity; 2.93 GiB already allocated; 2.06 MiB free; 2.93 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF